<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/00_environment_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import os
import torch

print("Python version:", sys.version)
print("Working directory:", os.getcwd())
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU not detected. Ensure runtime is set to GPU.")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Working directory: /content
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


In [3]:
from google.colab import drive
import os

drive.mount("/content/drive")

# Persistent directory for heavy artifacts (weights, extracted data)
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/DP-MMFL"
os.makedirs(os.path.join(DRIVE_PROJECT_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_ROOT, "data"), exist_ok=True)

print("Drive root ready:", DRIVE_PROJECT_ROOT)

MessageError: Error: credential propagation was unsuccessful

In [4]:
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)

DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/DP-MMFL"
os.makedirs(os.path.join(DRIVE_PROJECT_ROOT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT_ROOT, "data"), exist_ok=True)

print("Drive root ready:", DRIVE_PROJECT_ROOT)

Mounted at /content/drive
Drive root ready: /content/drive/MyDrive/DP-MMFL


In [5]:
# Faster disk operations for code and git commands
%cd /content

# If using a private repo, replace URL with https://<TOKEN>@github.com/<USER>/DP-MMFL.git
!git clone https://github.com/PreethamHD/DP-MMFL

%cd /content/DP-MMFL
!git status

/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 32 (delta 6), reused 26 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 4.59 KiB | 4.59 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/DP-MMFL
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [6]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 35.6 MB/s eta 0:00:00


In [7]:
import sys
import os

# Add src to the module search path
PROJECT_ROOT = "/content/DP-MMFL"
SRC_PATH = os.path.join(PROJECT_ROOT, "src")
if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

# Verify foundational third-party packages
import torch
import torchvision
import transformers
import pandas as pd
import numpy as np
import sklearn
import pydicom

# Verify custom package discovery
import dp_mmfl

print("--- Check Complete ---")
print("Third-party libraries loaded successfully.")
print(f"Package 'dp_mmfl' initialized. Version: {dp_mmfl.__version__}")

--- Check Complete ---
Third-party libraries loaded successfully.
Package 'dp_mmfl' initialized. Version: 0.1.0


In [8]:
%%writefile configs/base.yaml
project:
  name: DP-MMFL
  seed: 42

hardware:
  device: auto

data:
  primary_dataset: chexpert_plus
  external_dataset: iu_xray

model:
  multimodal: true

federated:
  enabled: false

privacy:
  enabled: false

imbalance:
  enabled: false

fairness:
  enabled: false

robust_aggregation:
  enabled: false

communication_efficiency:
  enabled: false

explainability:
  enabled: false

Overwriting configs/base.yaml


In [9]:
!mkdir -p src/dp_mmfl/utils
!touch src/dp_mmfl/utils/__init__.py

In [10]:
%%writefile src/dp_mmfl/utils/seed.py
import random
import numpy as np
import torch

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Overwriting src/dp_mmfl/utils/seed.py


In [11]:
%%writefile src/dp_mmfl/utils/environment.py
import platform
import sys
import torch

def get_environment_info():
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "pytorch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda if torch.cuda.is_available() else None,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }

Writing src/dp_mmfl/utils/environment.py


In [12]:
%%writefile docs/experiments.md
# DP-MMFL Experiments

## Experiment Naming

Format:
`EXP_<phase>_<description>`

Examples:
- `EXP_03_CENTRALIZED_IMAGE`
- `EXP_03_CENTRALIZED_MULTIMODAL`
- `EXP_04_FEDAVG`
- `EXP_05_FEDAVG_DP`
- `EXP_06_ADAPTIVE_DP`

## Results Policy

No experimental result may be entered unless the experiment has actually been executed.

Each experiment must record:
- Date
- Git commit
- Random seed
- Dataset version
- Dataset split
- Model
- Optimizer
- Learning rate
- Batch size
- Number of epochs
- Number of clients
- Federated rounds
- DP parameters
- Aggregation method
- Metricss

Overwriting docs/experiments.md


In [13]:
import sys
import os

repo_src = "/content/DP-MMFL/src"
if repo_src not in sys.path:
    sys.path.append(repo_src)

from dp_mmfl.utils.seed import set_seed
from dp_mmfl.utils.environment import get_environment_info

set_seed(42)
info = get_environment_info()

print("=" * 50)
print("DP-MMFL ENVIRONMENT CHECK")
print("=" * 50)
for key, value in info.items():
    print(f"{key:<16}: {value}")
print("=" * 50)
print("Environment setup complete.")

DP-MMFL ENVIRONMENT CHECK
python          : 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
platform        : Linux-6.6.122+-x86_64-with-glibc2.35
pytorch         : 2.11.0+cu128
cuda_available  : True
cuda_version    : 12.8
gpu             : Tesla T4
Environment setup complete.


In [14]:
!git status
!git add src/ configs/ docs/ data/
!git commit -m "Phase-0:setup project environment"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   configs/base.yaml
	modified:   docs/experiments.md
	modified:   src/dp_mmfl/utils/seed.py

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	src/dp_mmfl/__pycache__/
	src/dp_mmfl/utils/__pycache__/
	src/dp_mmfl/utils/environment.py

no changes added to commit (use "git add" and/or "git commit -a")
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@994ac202c055.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [15]:
!git config --global user.name "PreethamHD"
!git config --global user.email "preethamgowda837@gmail.com"

In [16]:
# Append to .gitignore if not present
!echo -e "\n__pycache__/\n*.py[cod]" >> .gitignore

In [18]:
# Stage your modified configs, docs, and python code
!git add configs/ docs/ src/ .gitignore

# Commit
!git commit -m "Phase-0:setup project environment"

# Push
!git push origin main

[main 2f50c1d] Phase-0:setup project environment
 9 files changed, 96 insertions(+), 6 deletions(-)
 create mode 100644 src/dp_mmfl/__pycache__/__init__.cpython-313.pyc
 create mode 100644 src/dp_mmfl/utils/__pycache__/__init__.cpython-313.pyc
 create mode 100644 src/dp_mmfl/utils/__pycache__/environment.cpython-313.pyc
 create mode 100644 src/dp_mmfl/utils/__pycache__/seed.cpython-313.pyc
 create mode 100644 src/dp_mmfl/utils/environment.py
Enumerating objects: 28, done.
Counting objects: 100% (26/26), done.
Delta compression using up to 2 threads
Compressing objects: 100% (16/16), done.
Writing objects: 100% (18/18), 3.27 KiB | 1.63 MiB/s, done.
Total 18 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/PreethamHD/DP-MMFL.git
   041fac5..2f50c1d  main -> main


In [19]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
